# 🌾 AgroMAE: Modelado Auto-Supervisado de Imágenes Satelitales para Uso de Suelos
## Inteligencia Artificial II — Segunda Entrega (Avance de Proyecto)

**Integrantes:**
* Andrés Felipe Alfonso Viviescas
* Andrés Felipe Higuera Grateron

---

### 🎯 Estructura del Cuaderno (Auditado y Alineado con la Rúbrica de 2da Entrega)
1. **Diagnóstico de Hardware y Optimización GPU L4:** Precisión mixta nativa (`bfloat16`/`float16`), `TF32` y *FlashAttention* (`scaled_dot_product_attention`).
2. **Sistema de Checkpoint Profesional (`CheckpointManager`):** Guardado en Google Drive en dos niveles (*Intra-Epoch* cada 200 pasos y *Epoch-Level*) con control exacto de época completada, pesos, momentos de AdamW, scheduler y *GradScaler*.
3. **Dataset Multiespectral EuroSAT (Sentinel-2, 13 Bandas):** Descarga verificada desde el repositorio oficial de Zenodo (`7711810`) / DFKI con respaldo automático, lectura `rasterio` y separación limpia de aumentaciones (solo en *Train*, nunca en *Val*).
4. **Arquitectura `AgroMAE` (Fase 1 - Pre-entrenamiento Auto-Supervisado):** Parcheado $8\times 8$, enmascaramiento aleatorio del 75%, ViT Encoder + Decoder ligero y pérdida MSE en parches ocultos.
5. **Gráficas de Convergencia y Reconstrucción Multiespectral (RGB + Falso Color NIR 8-4-3).**
6. **Experimento Preliminar Estratificado (Fase 2 - Escasez de Etiquetas al 10%):** Comparación cuantitativa (`Accuracy` y `F1-Score Macro`) entre entrenamiento desde cero (*Scratch*), *Linear Probing* y *Fine-Tuning* con tasas de aprendizaje diferenciales.

## 1. ⚙️ Diagnóstico de Hardware y Configuración de Aceleración (GPU L4)

In [ ]:
import os
import sys
import time
import math
import copy
import glob
import shutil
import urllib.request
import ssl
import zipfile
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, Subset

# Semilla para reproducibilidad científica
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DEVICE_TYPE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"📌 Dispositivo de cómputo: {device}")

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    total_vram = torch.cuda.get_device_properties(0).total_memory / (1024**3)
    print(f"🚀 GPU Detectada: {gpu_name} con {total_vram:.2f} GB de VRAM")
    
    # Activar optimizaciones para Tensor Cores de NVIDIA L4 (Ada Lovelace)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    
    AMP_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
    USE_SCALER = (AMP_DTYPE == torch.float16)
    print(f"⚡ Precisión Mixta seleccionada: {AMP_DTYPE} | GradScaler activo: {USE_SCALER}")
else:
    print("⚠️ ADVERTENCIA: No se detectó GPU. Selecciona GPU L4 en Entorno de Ejecución > Cambiar tipo de entorno.")
    AMP_DTYPE = torch.float32
    USE_SCALER = False

## 2. 📁 Montaje de Google Drive y Gestor de Checkpoints (`CheckpointManager`)

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_DIR = "/content/drive/MyDrive/AgroMAE_Proyecto"
except Exception:
    print("ℹ️ Ejecución fuera de Colab detectada. Usando directorio local.")
    DRIVE_DIR = "./AgroMAE_Local"

CHECKPOINT_DIR = os.path.join(DRIVE_DIR, "checkpoints")
LOGS_DIR = os.path.join(DRIVE_DIR, "logs")
os.makedirs(CHECKPOINT_DIR, exist_ok=True)
os.makedirs(LOGS_DIR, exist_ok=True)

class CheckpointManager:
    """
    Gestor de Checkpoints de doble nivel (Intra-Epoch y Epoch-Level).
    Distingue entre guardados a mitad de época (epoch_completed=False) y al final de época (epoch_completed=True),
    y es compatible con PyTorch 2.4+ (weights_only=False).
    """
    def __init__(self, checkpoint_dir):
        self.checkpoint_dir = checkpoint_dir
        self.last_path = os.path.join(checkpoint_dir, "checkpoint_last.pth")
        self.best_path = os.path.join(checkpoint_dir, "checkpoint_best.pth")

    def save(self, epoch, global_step, model, optimizer, scheduler=None, scaler=None,
             loss=None, best_val_loss=None, train_history=None, val_history=None,
             epoch_completed=False, is_best=False):
        state = {
            'epoch': int(epoch),
            'epoch_completed': bool(epoch_completed),
            'global_step': int(global_step),
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict() if scheduler else None,
            'scaler_state_dict': scaler.state_dict() if scaler else None,
            'loss': float(loss) if loss is not None else float('inf'),
            'best_val_loss': float(best_val_loss) if best_val_loss is not None else float('inf'),
            'train_history': list(train_history) if train_history is not None else [],
            'val_history': list(val_history) if val_history is not None else []
        }
        # Escritura atómica mediante archivo temporal para evitar corrupción si Colab se desconecta escribiendo
        tmp_path = self.last_path + ".tmp"
        torch.save(state, tmp_path)
        shutil.move(tmp_path, self.last_path)
        
        if is_best:
            shutil.copyfile(self.last_path, self.best_path)
            print(f"⭐ [RECORD] Nuevo mejor modelo guardado en Drive (Val MSE: {loss:.5f})")

    def load_latest(self, model, optimizer=None, scheduler=None, scaler=None, device="cuda"):
        if not os.path.exists(self.last_path):
            print("🚀 Sin checkpoint previo. Iniciando entrenamiento desde cero.")
            return 0, 0, float("inf"), [], []

        print(f"🔄 Cargando checkpoint desde: {self.last_path}")
        ckpt = torch.load(self.last_path, map_location=device, weights_only=False)
        model.load_state_dict(ckpt['model_state_dict'])
        if optimizer and ckpt.get('optimizer_state_dict'):
            optimizer.load_state_dict(ckpt['optimizer_state_dict'])
        if scheduler and ckpt.get('scheduler_state_dict'):
            scheduler.load_state_dict(ckpt['scheduler_state_dict'])
        if scaler and ckpt.get('scaler_state_dict'):
            scaler.load_state_dict(ckpt['scaler_state_dict'])

        saved_epoch = ckpt.get('epoch', 0)
        epoch_done = ckpt.get('epoch_completed', True)
        start_epoch = saved_epoch + 1 if epoch_done else saved_epoch
        start_step = ckpt.get('global_step', 0)
        best_loss = ckpt.get('best_val_loss', ckpt.get('loss', float("inf")))
        train_hist = ckpt.get('train_history', [])
        val_hist = ckpt.get('val_history', [])
        print(f"✅ Estado restaurado: reanudando en Época {start_epoch + 1}, Paso global {start_step}.")
        return start_epoch, start_step, best_loss, train_hist, val_hist

    def load_best_weights(self, model, device="cuda"):
        target = self.best_path if os.path.exists(self.best_path) else self.last_path
        if os.path.exists(target):
            ckpt = torch.load(target, map_location=device, weights_only=False)
            model.load_state_dict(ckpt['model_state_dict'])
            print(f"🏆 Pesos óptimos cargados desde: {target}")

## 3. 🛰️ Descarga Verificada y Carga de EuroSAT Multiespectral (Sentinel-2, 13 Bandas)
Descarga oficial desde **Zenodo Record `7711810` (`EuroSAT_MS.zip`)** con respaldo automático al servidor del **DFKI (`EuroSATallBands.zip`)**. Se crean instancias separadas para *Train* (con aumentaciones) y *Validation* (sin aumentaciones), usando partición estratificada por clase.

In [ ]:
try:
    import rasterio
    from sklearn.model_selection import StratifiedShuffleSplit
    from sklearn.metrics import f1_score, accuracy_score
except ImportError:
    !pip install -q rasterio scikit-learn
    import rasterio
    from sklearn.model_selection import StratifiedShuffleSplit
    from sklearn.metrics import f1_score, accuracy_score

LOCAL_DATA_DIR = "/content/eurosat_ms"
ZIP_PATH = "/content/EuroSAT_MS.zip"

# Verificar si ya existen los archivos .tif descomprimidos
existing_tifs = glob.glob(os.path.join(LOCAL_DATA_DIR, "**", "*.tif"), recursive=True)

if len(existing_tifs) < 1000:
    os.makedirs(LOCAL_DATA_DIR, exist_ok=True)
    urls = [
        "https://zenodo.org/records/7711810/files/EuroSAT_MS.zip?download=1",
        "https://madm.dfki.de/files/sentinel/EuroSATallBands.zip"
    ]
    downloaded = False
    for url in urls:
        print(f"📥 Intentando descargar EuroSAT Multiespectral (13 bandas) desde:\n   {url}")
        ret = os.system(f'wget --no-check-certificate -q --show-progress -O "{ZIP_PATH}" "{url}"')
        if ret == 0 and os.path.exists(ZIP_PATH) and os.path.getsize(ZIP_PATH) > 100_000_000:
            downloaded = True
            break
        else:
            print("⚠️ Falló el primer mirror, probando mirror de respaldo...")
            
    if not downloaded:
        raise RuntimeError("❌ No se pudo descargar EuroSAT_MS.zip. Verifica la conexión a internet de Colab.")
        
    print("📦 Descomprimiendo archivo en el SSD local de Colab...")
    with zipfile.ZipFile(ZIP_PATH, 'r') as zf:
        zf.extractall(LOCAL_DATA_DIR)
    os.remove(ZIP_PATH)
    print("✅ Extracción completada exitosamente.")
else:
    print(f"✅ Dataset ya disponible en disco local ({len(existing_tifs)} archivos .tif).")

class Sentinel2Dataset(Dataset):
    """
    Dataset para imágenes satelitales multiespectrales (13 bandas de Sentinel-2).
    Recibe una lista de rutas y etiquetas pre-calculadas para separar limpiamente Train y Val.
    """
    def __init__(self, file_paths, labels, classes, transform=False):
        self.file_paths = file_paths
        self.labels = labels
        self.classes = classes
        self.transform = transform

    def __len__(self):
        return len(self.file_paths)

    def __getitem__(self, idx):
        path = self.file_paths[idx]
        with rasterio.open(path) as src:
            img = src.read().astype(np.float32)  # (13, 64, 64)
        # Normalización radiométrica TOA de Sentinel-2
        img = np.clip(img / 10000.0, 0.0, 1.0)
        tensor = torch.from_numpy(img)
        
        if self.transform:
            if torch.rand(1) > 0.5:
                tensor = torch.flip(tensor, dims=[-1])  # Flip horizontal
            if torch.rand(1) > 0.5:
                tensor = torch.flip(tensor, dims=[-2])  # Flip vertical
        return tensor, self.labels[idx]

# Indexar todos los archivos .tif y sus clases
all_files = sorted(glob.glob(os.path.join(LOCAL_DATA_DIR, "**", "*.tif"), recursive=True))
assert len(all_files) > 0, "❌ No se encontraron archivos .tif en LOCAL_DATA_DIR."

class_names = sorted(list(set([os.path.basename(os.path.dirname(f)) for f in all_files])))
class_to_idx = {c: i for i, c in enumerate(class_names)}
all_labels = np.array([class_to_idx[os.path.basename(os.path.dirname(f))] for f in all_files])

# Partición Estratificada 85% Train / 15% Val
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.15, random_state=SEED)
train_idx, val_idx = next(sss.split(all_files, all_labels))

train_files = [all_files[i] for i in train_idx]
train_labels = all_labels[train_idx]
val_files = [all_files[i] for i in val_idx]
val_labels = all_labels[val_idx]

train_ds = Sentinel2Dataset(train_files, train_labels, class_names, transform=True)
val_ds = Sentinel2Dataset(val_files, val_labels, class_names, transform=False)

BATCH_SIZE = 128
NUM_WORKERS = 2
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)

print(f"📊 Total imágenes: {len(all_files)} | Clases ({len(class_names)}): {class_names}")
print(f"✅ Partición Estratificada -> Train: {len(train_ds)} imgs | Val: {len(val_ds)} imgs")

## 4. 🔬 Arquitectura `AgroMAE` (Masked Autoencoder Multiespectral de 13 Bandas)

In [ ]:
def get_2d_sincos_pos_embed(embed_dim, grid_size):
    grid_h = np.arange(grid_size, dtype=np.float32)
    grid_w = np.arange(grid_size, dtype=np.float32)
    grid = np.meshgrid(grid_w, grid_h)
    grid = np.stack(grid, axis=0).reshape([2, 1, grid_size, grid_size])
    
    def get_1d(dim, pos):
        omega = 1. / (10000 ** (np.arange(dim // 2, dtype=np.float64) / (dim / 2.)))
        out = np.einsum('m,d->md', pos.reshape(-1), omega)
        return np.concatenate([np.sin(out), np.cos(out)], axis=1)

    return np.concatenate([get_1d(embed_dim // 2, grid[0]), get_1d(embed_dim // 2, grid[1])], axis=1)

class TransformerBlock(nn.Module):
    def __init__(self, dim, num_heads, mlp_ratio=4.0):
        super().__init__()
        self.norm1 = nn.LayerNorm(dim)
        self.num_heads = num_heads
        self.head_dim = dim // num_heads
        self.qkv = nn.Linear(dim, dim * 3, bias=True)
        self.proj = nn.Linear(dim, dim)
        self.norm2 = nn.LayerNorm(dim)
        self.mlp = nn.Sequential(
            nn.Linear(dim, int(dim * mlp_ratio)),
            nn.GELU(),
            nn.Linear(int(dim * mlp_ratio), dim)
        )

    def forward(self, x):
        B, N, C = x.shape
        qkv = self.qkv(self.norm1(x)).reshape(B, N, 3, self.num_heads, self.head_dim).permute(2, 0, 3, 1, 4)
        attn = F.scaled_dot_product_attention(qkv[0], qkv[1], qkv[2]).transpose(1, 2).reshape(B, N, C)
        x = x + self.proj(attn)
        x = x + self.mlp(self.norm2(x))
        return x

class AgroMAE(nn.Module):
    def __init__(self, img_size=64, patch_size=8, in_chans=13,
                 embed_dim=256, depth=6, num_heads=8,
                 decoder_embed_dim=128, decoder_depth=3, decoder_num_heads=4,
                 mask_ratio=0.75):
        super().__init__()
        self.img_size = img_size
        self.patch_size = patch_size
        self.in_chans = in_chans
        self.grid_size = img_size // patch_size
        self.num_patches = self.grid_size ** 2
        self.mask_ratio = mask_ratio
        
        self.patch_embed = nn.Conv2d(in_chans, embed_dim, kernel_size=patch_size, stride=patch_size)
        self.pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, embed_dim), requires_grad=False)
        self.blocks = nn.ModuleList([TransformerBlock(embed_dim, num_heads) for _ in range(depth)])
        self.norm = nn.LayerNorm(embed_dim)
        
        self.decoder_embed = nn.Linear(embed_dim, decoder_embed_dim, bias=True)
        self.mask_token = nn.Parameter(torch.zeros(1, 1, decoder_embed_dim))
        self.decoder_pos_embed = nn.Parameter(torch.zeros(1, self.num_patches, decoder_embed_dim), requires_grad=False)
        self.decoder_blocks = nn.ModuleList([TransformerBlock(decoder_embed_dim, decoder_num_heads) for _ in range(decoder_depth)])
        self.decoder_norm = nn.LayerNorm(decoder_embed_dim)
        self.decoder_pred = nn.Linear(decoder_embed_dim, patch_size * patch_size * in_chans, bias=True)
        self.initialize_weights()

    def initialize_weights(self):
        self.pos_embed.data.copy_(torch.from_numpy(get_2d_sincos_pos_embed(self.pos_embed.shape[-1], self.grid_size)).float().unsqueeze(0))
        self.decoder_pos_embed.data.copy_(torch.from_numpy(get_2d_sincos_pos_embed(self.decoder_pos_embed.shape[-1], self.grid_size)).float().unsqueeze(0))
        w = self.patch_embed.weight.data
        torch.nn.init.xavier_uniform_(w.view([w.shape[0], -1]))
        torch.nn.init.normal_(self.mask_token, std=.02)

    def patchify(self, imgs):
        p, h, c = self.patch_size, self.grid_size, self.in_chans
        x = imgs.reshape(imgs.shape[0], c, h, p, h, p)
        return torch.einsum('nchpwq->nhwpqc', x).reshape(imgs.shape[0], h * h, p * p * c)

    def unpatchify(self, x):
        p, h, c = self.patch_size, self.grid_size, self.in_chans
        x = x.reshape(x.shape[0], h, h, p, p, c)
        return torch.einsum('nhwpqc->nchpwq', x).reshape(x.shape[0], c, h * p, h * p)

    def random_masking(self, x, mask_ratio):
        N, L, D = x.shape
        len_keep = int(L * (1 - mask_ratio))
        noise = torch.rand(N, L, device=x.device)
        ids_shuffle = torch.argsort(noise, dim=1)
        ids_restore = torch.argsort(ids_shuffle, dim=1)
        ids_keep = ids_shuffle[:, :len_keep]
        x_masked = torch.gather(x, dim=1, index=ids_keep.unsqueeze(-1).repeat(1, 1, D))
        mask = torch.ones([N, L], device=x.device)
        mask[:, :len_keep] = 0
        mask = torch.gather(mask, dim=1, index=ids_restore)
        return x_masked, mask, ids_restore

    def forward_encoder(self, x, mask_ratio):
        x = self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos_embed
        x, mask, ids_restore = self.random_masking(x, mask_ratio)
        for blk in self.blocks:
            x = blk(x)
        return self.norm(x), mask, ids_restore

    def forward_decoder(self, x, ids_restore):
        x = self.decoder_embed(x)
        mask_tokens = self.mask_token.repeat(x.shape[0], ids_restore.shape[1] - x.shape[1], 1)
        x_ = torch.cat([x, mask_tokens], dim=1)
        x = torch.gather(x_, dim=1, index=ids_restore.unsqueeze(-1).repeat(1, 1, x.shape[2])) + self.decoder_pos_embed
        for blk in self.decoder_blocks:
            x = blk(x)
        return self.decoder_pred(self.decoder_norm(x))

    def forward_loss(self, imgs, pred, mask):
        target = self.patchify(imgs)
        loss = ((pred - target) ** 2).mean(dim=-1)
        return (loss * mask).sum() / mask.sum()

    def forward(self, imgs, mask_ratio=None):
        if mask_ratio is None:
            mask_ratio = self.mask_ratio
        latent, mask, ids_restore = self.forward_encoder(imgs, mask_ratio)
        pred = self.forward_decoder(latent, ids_restore)
        loss = self.forward_loss(imgs, pred, mask)
        return loss, pred, mask

model = AgroMAE().to(device)
# Sanity check instantáneo
with torch.no_grad():
    _dummy = torch.rand(2, 13, 64, 64, device=device)
    _loss, _pred, _mask = model(_dummy)
print(f"⭐ AgroMAE verificado | Parámetros entrenables: {sum(p.numel() for p in model.parameters() if p.requires_grad):,} | Test Loss: {_loss.item():.4f}")

## 5. 🚀 Fase 1: Pre-Entrenamiento Auto-Supervisado con Checkpointing de Doble Nivel

In [ ]:
TOTAL_EPOCHS = 35
BASE_LR = 1.5e-4
SAVE_EVERY_STEPS = 200

optimizer = torch.optim.AdamW(model.parameters(), lr=BASE_LR, betas=(0.9, 0.95), weight_decay=0.05)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TOTAL_EPOCHS, eta_min=1e-6)
scaler = torch.amp.GradScaler(DEVICE_TYPE, enabled=USE_SCALER)
ckpt_manager = CheckpointManager(checkpoint_dir=CHECKPOINT_DIR)

start_epoch, global_step, best_val_loss, train_losses, val_losses = ckpt_manager.load_latest(
    model, optimizer, scheduler, scaler, device
)

def validate_mae(model, loader):
    model.eval()
    total_loss = 0.0
    with torch.no_grad():
        for imgs, _ in loader:
            imgs = imgs.to(device, non_blocking=True)
            with torch.amp.autocast(DEVICE_TYPE, dtype=AMP_DTYPE, enabled=(DEVICE_TYPE == "cuda")):
                loss, _, _ = model(imgs)
            total_loss += loss.item()
    return total_loss / len(loader)

for epoch in range(start_epoch, TOTAL_EPOCHS):
    model.train()
    epoch_loss = 0.0
    t0 = time.time()
    for imgs, _ in train_loader:
        imgs = imgs.to(device, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)
        with torch.amp.autocast(DEVICE_TYPE, dtype=AMP_DTYPE, enabled=(DEVICE_TYPE == "cuda")):
            loss, _, _ = model(imgs)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        epoch_loss += loss.item()
        global_step += 1
        
        # Checkpoint Intra-Epoch (epoch_completed=False para no saltar época si se interrumpe)
        if global_step % SAVE_EVERY_STEPS == 0:
            ckpt_manager.save(
                epoch, global_step, model, optimizer, scheduler, scaler,
                loss=loss.item(), best_val_loss=best_val_loss,
                train_history=train_losses, val_history=val_losses,
                epoch_completed=False, is_best=False
            )

    scheduler.step()
    tr_loss = epoch_loss / len(train_loader)
    va_loss = validate_mae(model, val_loader)
    psnr = 10 * math.log10(1.0 / max(va_loss, 1e-8))
    train_losses.append(tr_loss)
    val_losses.append(va_loss)
    
    is_best = va_loss < best_val_loss
    if is_best:
        best_val_loss = va_loss

    print(f"Epoch [{epoch+1:02d}/{TOTAL_EPOCHS}] | Train MSE: {tr_loss:.5f} | Val MSE: {va_loss:.5f} | PSNR: {psnr:.2f} dB | {time.time()-t0:.1f}s")
    # Checkpoint Fin de Época (epoch_completed=True)
    ckpt_manager.save(
        epoch, global_step, model, optimizer, scheduler, scaler,
        loss=va_loss, best_val_loss=best_val_loss,
        train_history=train_losses, val_history=val_losses,
        epoch_completed=True, is_best=is_best
    )

## 6. 📈 Resultados Preliminares — Parte A: Curva de Convergencia y Reconstrucción Multiespectral

In [ ]:
# Cargar el mejor checkpoint obtenido antes de visualizar y evaluar
ckpt_manager.load_best_weights(model, device=device)

if len(train_losses) > 0:
    epochs_range = range(1, len(train_losses) + 1)
    psnr_curve = [10 * math.log10(1.0 / max(v, 1e-8)) for v in val_losses]
    
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
    ax1.plot(epochs_range, train_losses, 'o-', label='Train MSE (Parches Ocultos)', color='#1f77b4')
    ax1.plot(epochs_range, val_losses, 's-', label='Val MSE (Parches Ocultos)', color='#ff7f0e')
    ax1.set_title('Convergencia del Pre-entrenamiento Auto-Supervisado (AgroMAE)')
    ax1.set_xlabel('Época')
    ax1.set_ylabel('Error Cuadrático Medio (MSE)')
    ax1.grid(True, linestyle='--', alpha=0.5)
    ax1.legend()
    
    ax2.plot(epochs_range, psnr_curve, 'd-', color='#2ca02c', label='Val PSNR (dB)')
    ax2.set_title('Relación Señal a Ruido Pico (PSNR Espectral en 13 Bandas)')
    ax2.set_xlabel('Época')
    ax2.set_ylabel('PSNR (dB)')
    ax2.grid(True, linestyle='--', alpha=0.5)
    ax2.legend()
    plt.tight_layout()
    plt.show()

def visualize_reconstruction(model, dataloader, num_samples=3):
    model.eval()
    imgs, _ = next(iter(dataloader))
    imgs = imgs[:num_samples].to(device)
    with torch.no_grad():
        with torch.amp.autocast(DEVICE_TYPE, dtype=AMP_DTYPE, enabled=(DEVICE_TYPE == "cuda")):
            _, pred, mask = model(imgs)
        pred_imgs = torch.clamp(model.unpatchify(pred), 0.0, 1.0)
        mask_exp = mask.unsqueeze(-1).repeat(1, 1, model.patch_size ** 2 * model.in_chans)
        mask_2d = model.unpatchify(mask_exp)
        masked_inputs = imgs * (1.0 - mask_2d)

    imgs = imgs.cpu().float().numpy()
    masked_inputs = masked_inputs.cpu().float().numpy()
    pred_imgs = pred_imgs.cpu().float().numpy()

    fig, axes = plt.subplots(num_samples, 4, figsize=(16, 4 * num_samples))
    if num_samples == 1:
        axes = np.expand_dims(axes, 0)
        
    for i in range(num_samples):
        axes[i, 0].imshow(np.clip(np.stack([imgs[i, 3], imgs[i, 2], imgs[i, 1]], axis=-1) * 3.5, 0, 1))
        axes[i, 0].set_title("Original (RGB Sentinel-2)")
        axes[i, 1].imshow(np.clip(np.stack([masked_inputs[i, 3], masked_inputs[i, 2], masked_inputs[i, 1]], axis=-1) * 3.5, 0, 1))
        axes[i, 1].set_title("Entrada al Encoder (75% Oculto)")
        axes[i, 2].imshow(np.clip(np.stack([pred_imgs[i, 3], pred_imgs[i, 2], pred_imgs[i, 1]], axis=-1) * 3.5, 0, 1))
        axes[i, 2].set_title("Reconstrucción AgroMAE (RGB)")
        axes[i, 3].imshow(np.clip(np.stack([pred_imgs[i, 7], pred_imgs[i, 3], pred_imgs[i, 2]], axis=-1) * 3.0, 0, 1))
        axes[i, 3].set_title("Reconstrucción Falso Color NIR (8-4-3)")
        for j in range(4):
            axes[i, j].axis('off')
    plt.tight_layout()
    plt.show()

visualize_reconstruction(model, val_loader, num_samples=3)

## 7. 🏆 Resultados Preliminares — Parte B: Experimento Estratificado en Escasez de Etiquetas (10% Datos)
Evaluamos la clasificación de las 10 coberturas de suelo usando una **muestra estratificada del 10% de los datos de entrenamiento (~2,295 imágenes)** bajo tres esquemas:
1. **Baseline Desde Cero (*Scratch*):** Misma arquitectura inicializada con pesos aleatorios.
2. **AgroMAE (*Linear Probing*):** Encoder pre-entrenado congelado (solo se entrena el cabezal clasificador).
3. **AgroMAE (*Fine-Tuning Completo*):** Ajuste fino de extremo a extremo con tasa de aprendizaje diferencial.

In [ ]:
class AgroMAEClassifier(nn.Module):
    def __init__(self, mae_encoder, num_classes=10, freeze_encoder=False):
        super().__init__()
        self.patch_embed = copy.deepcopy(mae_encoder.patch_embed)
        self.pos_embed = copy.deepcopy(mae_encoder.pos_embed)
        self.blocks = copy.deepcopy(mae_encoder.blocks)
        self.norm = copy.deepcopy(mae_encoder.norm)
        
        if freeze_encoder:
            for param in self.patch_embed.parameters():
                param.requires_grad = False
            for param in self.blocks.parameters():
                param.requires_grad = False
            for param in self.norm.parameters():
                param.requires_grad = False

        embed_dim = mae_encoder.pos_embed.shape[-1]
        self.head = nn.Sequential(
            nn.LayerNorm(embed_dim),
            nn.Dropout(0.15),
            nn.Linear(embed_dim, 128),
            nn.GELU(),
            nn.Linear(128, num_classes)
        )

    def forward(self, x):
        x = self.patch_embed(x).flatten(2).transpose(1, 2) + self.pos_embed
        for blk in self.blocks:
            x = blk(x)
        x = self.norm(x).mean(dim=1)  # Global Average Pooling
        return self.head(x)

# Selección Estratificada del 10% del conjunto de entrenamiento (igual proporción de las 10 clases)
sss_10 = StratifiedShuffleSplit(n_splits=1, train_size=0.10, random_state=SEED)
idx_10pct, _ = next(sss_10.split(train_files, train_labels))
subset_10pct = Subset(train_ds, idx_10pct)
loader_10pct = DataLoader(subset_10pct, batch_size=64, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True)

def train_and_eval_classifier(clf_model, epochs=18, encoder_lr=1.5e-4, head_lr=1e-3, name="Model"):
    clf_model = clf_model.to(device)
    # Tasas de aprendizaje diferenciales para proteger los pesos pre-entrenados
    encoder_params = [p for n, p in clf_model.named_parameters() if "head" not in n and p.requires_grad]
    head_params = [p for n, p in clf_model.named_parameters() if "head" in n and p.requires_grad]
    
    param_groups = [{'params': head_params, 'lr': head_lr}]
    if len(encoder_params) > 0:
        param_groups.append({'params': encoder_params, 'lr': encoder_lr})
        
    opt = torch.optim.AdamW(param_groups, weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs, eta_min=1e-5)
    clf_scaler = torch.amp.GradScaler(DEVICE_TYPE, enabled=USE_SCALER)
    criterion = nn.CrossEntropyLoss()
    
    for ep in range(epochs):
        clf_model.train()
        for x, y in loader_10pct:
            x, y = x.to(device, non_blocking=True), y.to(device, non_blocking=True, dtype=torch.long)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast(DEVICE_TYPE, dtype=AMP_DTYPE, enabled=(DEVICE_TYPE == "cuda")):
                logits = clf_model(x)
                loss = criterion(logits, y)
            clf_scaler.scale(loss).backward()
            clf_scaler.step(opt)
            clf_scaler.update()
        sched.step()

    # Evaluación en el conjunto de validación completo (4,050 imágenes sin aumentación)
    clf_model.eval()
    all_preds, all_targets = [], []
    with torch.no_grad():
        for x, y in val_loader:
            x = x.to(device, non_blocking=True)
            with torch.amp.autocast(DEVICE_TYPE, dtype=AMP_DTYPE, enabled=(DEVICE_TYPE == "cuda")):
                preds = clf_model(x).argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_targets.extend(y.numpy())

    acc = accuracy_score(all_targets, all_preds) * 100
    f1 = f1_score(all_targets, all_preds, average='macro') * 100
    print(f"📊 [{name}] -> Val Accuracy: {acc:.2f}% | F1-Score Macro: {f1:.2f}%")
    return acc, f1

print(f"🔬 Ejecutando experimento estratificado con el 10% de etiquetas ({len(subset_10pct)} muestras)...")

# 1. Baseline Desde Cero (Scratch)
untrained_mae = AgroMAE().to(device)
clf_scratch = AgroMAEClassifier(untrained_mae, num_classes=10, freeze_encoder=False)
acc_scratch, f1_scratch = train_and_eval_classifier(
    clf_scratch, epochs=18, encoder_lr=3e-4, head_lr=3e-4, name="1. Baseline Desde Cero (Scratch)"
)

# 2. AgroMAE Linear Probing (Encoder Congelado)
clf_linear = AgroMAEClassifier(model, num_classes=10, freeze_encoder=True)
acc_linear, f1_linear = train_and_eval_classifier(
    clf_linear, epochs=18, encoder_lr=0.0, head_lr=1e-3, name="2. AgroMAE (Linear Probing)"
)

# 3. AgroMAE Fine-Tuning Completo
clf_finetune = AgroMAEClassifier(model, num_classes=10, freeze_encoder=False)
acc_ft, f1_ft = train_and_eval_classifier(
    clf_finetune, epochs=18, encoder_lr=1.5e-4, head_lr=1e-3, name="3. AgroMAE (Fine-Tuning Completo)"
)

# Gráfica comparativa de los 3 modelos
labels = ['Accuracy (%)', 'F1-Score Macro (%)']
scratch_vals = [acc_scratch, f1_scratch]
linear_vals = [acc_linear, f1_linear]
ft_vals = [acc_ft, f1_ft]

x = np.arange(len(labels))
width = 0.25
fig, ax = plt.subplots(figsize=(9, 5))
r1 = ax.bar(x - width, scratch_vals, width, label='Desde Cero (Sin Pre-entrenar)', color='#7f8c8d')
r2 = ax.bar(x, linear_vals, width, label='AgroMAE (Linear Probing)', color='#2980b9')
r3 = ax.bar(x + width, ft_vals, width, label='AgroMAE (Fine-Tuning SSL 75%)', color='#27ae60')
ax.set_ylabel('Porcentaje (%)')
ax.set_title('Desempeño con Escasez de Datos (10% de Etiquetas Estratificadas)')
ax.set_xticks(x)
ax.set_xticklabels(labels)
ax.set_ylim(0, 108)
ax.legend()
ax.bar_label(r1, fmt='%.1f%%', padding=3)
ax.bar_label(r2, fmt='%.1f%%', padding=3)
ax.bar_label(r3, fmt='%.1f%%', padding=3)
plt.grid(axis='y', linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()